# 08 · Вмешательства в точке вызова: приблизиться к LoRA без обучения весов

**Цель** (JOURNAL, 2026-10-27/30). Диагностика 07: эффект LoRA передаётся через одну позицию — точку
вызова — на слоях 22+, а изменение, которое делает LoRA, — в основном постоянный сдвиг плюс линейная
функция состояния. Методы здесь — поправка в этой точке:

| метод | как | обучение |
|---|---|---|
| `linmap_lora` | аффинная поправка, подогнанная в закрытой форме к Δh LoRA | только у учителя (LoRA) |
| `linmap` | то же без учителя: цель — центроид верных состояний базы | **нет** |
| `reft` | LoReFT (2404.03592) в точке вызова | градиент только по вмешательству |
| `cast` | CAST (2409.05907): условие по PC1, вектор поведения | нет |

Данные — те же 110 промптов, что у вектора и LoRA; выбор на val; проверка на новом наборе переноса
(seed 202). Плюс проверка формы из 07 на слоях 22–26. Результат **разведочный**.

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [2]:
cfg = ExperimentConfig.make(
    name="act-methods-qwen3-1.7b", hypothesis="explore", notebook="notebooks/08_activation_methods.ipynb",
    notes="Вмешательства в точке вызова: linmap (с LoRA-учителем и без), LoReFT, CAST против LoRA.",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",
        "data.n_tasks": 160,
        "data.generator": "v3",
        "eval.methods": ["baseline", "steering", "steering_gated", "lora",
                         "linmap_lora", "linmap", "reft", "cast"],
        "steer.alphas": [0.25, 0.5, 1.0, 2.0, 4.0],
        "episode.on_val": False,
        "transfer.seed": 202,
        "transfer.episode_methods": ["baseline", "lora", "linmap_lora", "linmap", "reft"],
    },
)
exp = Experiment(cfg)
exp.run("fit")
exp.results_table("val")

/Users/admin/Desktop/steering-tool-feedback/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[20:47:04] run 20261030-204700_act-methods-qwen3-1.7b_74c4d6  ->  results/runs/20261030-204700_act-methods-qwen3-1.7b_74c4d6
[20:47:04] data: {'train': 79, 'val': 31, 'test': 50}  types={'invariant': 41, 'changing': 112, 'stress': 7}  positions={'early': 114, 'late': 46}


baseline/train:   0%|          | 0/158 [00:00<?, ?it/s]

[20:47:04] loading model Qwen/Qwen3-1.7B …


W1030 20:47:11.339000 2035 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

... (8,273 outputs left out)

cast/val:   0%|          | 0/62 [00:00<?, ?it/s]

cast/val:  82%|████████▏ | 51/62 [00:00<00:00, 503.25it/s]

[22:23:35] val: baseline 0.621; steering 0.655; steering_gated 0.638; lora 0.948; linmap_lora 0.845; linmap 0.862; reft 0.741; cast 0.621


,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.621,0.621,0.526,0.4,NaN,,NaN,NaN,None,None,,0.50,831,10.0,400.6
1,steering,0.655,0.655,0.632,0.4,0.034,"[+0.000, +0.086]",2.0,0.0,None,None,,0.50,831,10.0,410.2
2,steering_gated,0.638,0.638,0.579,0.4,0.017,"[+0.000, +0.052]",1.0,0.0,None,None,,0.50,831,10.0,408.6
3,lora,0.948,0.948,1.000,0.8,0.328,"[+0.190, +0.466]",19.0,0.0,None,None,,0.00,831,10.0,299.0
4,linmap_lora,0.845,0.845,0.789,0.7,0.224,"[+0.086, +0.379]",18.0,5.0,None,None,,0.50,831,10.0,425.8
5,linmap,0.862,0.862,0.842,0.6,0.241,"[+0.121, +0.379]",16.0,2.0,None,None,,0.50,831,10.0,410.6
6,reft,0.741,0.741,0.684,0.5,0.121,"[-0.017, +0.259]",13.0,6.0,None,None,,0.75,831,10.0,442.9
7,cast,0.621,0.621,0.526,0.4,0.000,"[+0.000, +0.000]",0.0,0.0,None,None,,0.50,831,10.0,440.6


### Подбор на val

In [3]:
for m in ("linmap_lora", "linmap", "reft", "cast"):
    t = exp.fit_table(m)
    if t.empty:
        print(m, exp.state["fitted"][m].get("reason")); continue
    display(Markdown(f"**{m}** — лучшие конфигурации на val"))
    cols = [c for c in ("layer", "rank", "alpha", "cond_layer", "fire_rate", "train_r2", "train_seconds",
                        "success", "sensitivity", "specificity", "spec_drop", "fixed", "broken") if c in t]
    display(t.sort_values("success", ascending=False)[cols].head(8).round(3))

**linmap_lora** — лучшие конфигурации на val

,layer,rank,alpha,train_r2,success,sensitivity,specificity,spec_drop,fixed,broken
53,26,16,1.5,0.963,0.845,0.789,0.7,-0.3,18,5
44,24,16,1.5,0.944,0.828,0.632,0.8,-0.4,16,4
14,18,8,1.5,0.839,0.793,0.789,0.5,-0.1,10,0
17,18,16,1.5,0.891,0.793,0.789,0.5,-0.1,10,0
5,16,8,1.5,0.865,0.776,0.789,0.4,0.0,9,0
41,24,8,1.5,0.914,0.776,0.579,0.7,-0.3,13,4
11,18,4,1.5,0.774,0.776,0.789,0.5,-0.1,9,0
25,20,16,1.0,0.911,0.776,0.789,0.5,-0.1,9,0


**linmap** — лучшие конфигурации на val

,layer,rank,alpha,train_r2,success,sensitivity,specificity,spec_drop,fixed,broken
17,18,16,1.5,0.568,0.862,0.789,0.8,-0.4,17,3
14,18,8,1.5,0.547,0.862,0.842,0.6,-0.2,16,2
11,18,4,1.5,0.502,0.845,0.895,0.5,-0.1,15,2
26,20,16,1.5,0.561,0.828,0.737,0.6,-0.2,15,3
16,18,16,1.0,0.568,0.810,0.842,0.5,-0.1,11,0
20,20,4,1.5,0.492,0.810,0.842,0.5,-0.1,12,1
5,16,8,1.5,0.516,0.810,0.789,0.6,-0.2,12,1
8,16,16,1.5,0.531,0.810,0.789,0.6,-0.2,12,1


**reft** — лучшие конфигурации на val

,layer,rank,alpha,train_r2,train_seconds,success,sensitivity,specificity,spec_drop,fixed,broken
1,22,4,1.0,None,590.4,0.741,0.684,0.5,-0.1,13,6
0,18,4,1.0,None,659.1,0.569,0.421,0.4,0.0,4,7
2,26,4,1.0,None,468.1,0.534,0.474,0.3,0.1,7,12


**cast** — лучшие конфигурации на val

,layer,alpha,cond_layer,fire_rate,success,sensitivity,specificity,spec_drop,fixed,broken
0,16,0.5,17,0.919,0.621,0.526,0.4,0.0,0,0
1,16,1.0,17,0.919,0.621,0.526,0.4,0.0,1,1
2,16,2.0,17,0.919,0.621,0.526,0.4,0.0,1,1
6,24,0.5,17,0.919,0.621,0.526,0.4,0.0,0,0
7,24,1.0,17,0.919,0.621,0.526,0.4,0.0,0,0
3,20,0.5,17,0.919,0.603,0.474,0.4,0.0,0,1
4,20,1.0,17,0.919,0.603,0.474,0.4,0.0,0,1
5,20,2.0,17,0.919,0.603,0.474,0.4,0.0,0,1


### Проверка формы на слоях 22–26 (диагностика 07, без подстановки)

In [4]:
diag = exp.lora_diagnostics(patch=False, form_layers=[22, 23, 24, 25, 26])
pd.DataFrame(diag["form_check"]).round(3)

lora/train:   0%|          | 0/158 [00:00<?, ?it/s]

lora diagnostics:   0%|          | 0/156 [00:00<?, ?it/s]

lora diagnostics:   1%|          | 1/156 [00:02<05:50,  2.26s/it]

lora diagnostics:   1%|▏         | 2/156 [00:04<05:56,  2.31s/it]

lora diagnostics:   2%|▏         | 3/156 [00:06<05:54,  2.32s/it]

lora diagnostics:   3%|▎         | 4/156 [00:09<06:00,  2.37s/it]

... (150 outputs left out)

lora diagnostics:  99%|█████████▉| 155/156 [06:36<00:02,  2.48s/it]

lora diagnostics: 100%|██████████| 156/156 [06:39<00:00,  2.50s/it]

[22:40:36] lora diagnostics: 156 промптов train, решения LoRA и базы расходятся в 63; первый токен = полный скоринг в 100%


,layer,form,agree_with_lora,fixed_like_lora,kept_where_agree,n
0,22,const,0.609,0.048,0.989,156
1,22,probe,0.622,0.159,0.935,156
2,22,linear,0.853,0.683,0.968,156
3,22,base,0.596,0.000,1.000,156
4,23,const,0.609,0.095,0.957,156
5,23,probe,0.641,0.222,0.925,156
6,23,linear,0.885,0.762,0.968,156
7,23,base,0.596,0.000,1.000,156
8,24,const,0.609,0.127,0.935,156
9,24,probe,0.647,0.238,0.925,156


### Новый набор переноса (seed 202): знакомые и новые типы требований

In [5]:
exp.transfer()
for g in ("seen", "unseen"):
    display(Markdown(f"**{g}**"))
    display(exp.results_table(f"transfer:{g}"))

baseline/transfer:seen:   0%|          | 0/100 [00:00<?, ?it/s]

baseline/transfer:seen:   1%|          | 1/100 [00:00<00:33,  2.97it/s]

baseline/transfer:seen:   2%|▏         | 2/100 [00:00<00:33,  2.95it/s]

baseline/transfer:seen:   3%|▎         | 3/100 [00:01<00:36,  2.62it/s]

baseline/transfer:seen:   4%|▍         | 4/100 [00:01<00:37,  2.55it/s]

baseline/transfer:seen:   5%|▌         | 5/100 [00:01<00:38,  2.47it/s]

... (2,623 outputs left out)

**seen**

,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.60,0.60,0.270,0.615,NaN,,NaN,NaN,0.56,NaN,,None,834,10.0,420.9
1,steering,0.63,0.63,0.216,0.692,0.03,"[-0.040, +0.110]",8.0,5.0,NaN,NaN,,None,834,10.0,414.7
2,steering_gated,0.64,0.64,0.270,0.692,0.04,"[-0.010, +0.110]",5.0,1.0,NaN,NaN,,None,834,10.0,416.1
3,lora,0.96,0.96,0.892,1.000,0.36,"[+0.260, +0.460]",38.0,2.0,0.84,0.28,"[+0.170, +0.390]",None,834,10.0,428.2
4,linmap_lora,0.73,0.73,0.459,0.692,0.13,"[+0.020, +0.240]",26.0,13.0,0.68,0.12,"[+0.020, +0.230]",None,834,10.0,424.0
5,linmap,0.85,0.85,0.595,1.000,0.25,"[+0.150, +0.360]",32.0,7.0,0.68,0.12,"[+0.010, +0.230]",None,834,10.0,426.9
6,reft,0.55,0.55,0.108,0.462,-0.05,"[-0.160, +0.060]",18.0,23.0,0.53,-0.03,"[-0.150, +0.080]",None,834,10.0,426.1
7,cast,0.60,0.60,0.297,0.462,0.00,"[-0.060, +0.050]",3.0,3.0,NaN,NaN,,None,834,10.0,850.7


**unseen**

,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.45,0.45,0.118,0.250,NaN,,NaN,NaN,0.38,NaN,,None,869,10.0,424.1
1,steering,0.54,0.54,0.235,0.438,0.09,"[+0.020, +0.170]",12.0,3.0,NaN,NaN,,None,869,10.0,409.4
2,steering_gated,0.48,0.48,0.118,0.312,0.03,"[-0.020, +0.080]",5.0,2.0,NaN,NaN,,None,869,10.0,415.5
3,lora,0.85,0.85,0.618,0.875,0.40,"[+0.290, +0.510]",42.0,2.0,0.79,0.41,"[+0.290, +0.520]",None,869,10.0,366.9
4,linmap_lora,0.78,0.78,0.441,0.812,0.33,"[+0.210, +0.450]",41.0,8.0,0.74,0.36,"[+0.240, +0.480]",None,869,10.0,355.9
5,linmap,0.61,0.61,0.294,0.438,0.16,"[+0.090, +0.240]",17.0,1.0,0.39,0.01,"[-0.070, +0.100]",None,869,10.0,357.4
6,reft,0.53,0.53,0.382,0.312,0.08,"[-0.010, +0.170]",14.0,6.0,0.52,0.14,"[+0.040, +0.240]",None,869,10.0,358.3
7,cast,0.49,0.49,0.176,0.375,0.04,"[-0.020, +0.100]",6.0,2.0,NaN,NaN,,None,869,10.0,764.9


In [6]:
rows = []
for g in ("seen", "unseen"):
    for name, c in exp.state["transfer"][g]["comparisons"].items():
        rows.append({"group": g, "comparison": name, "diff": c["mean_diff"],
                     "ci": f"[{c['ci_low']:+.3f}, {c['ci_high']:+.3f}]", "excludes_zero": c["excludes_zero"]})
pd.DataFrame(rows)

,group,comparison,diff,ci,excludes_zero
0,seen,steering_vs_steering_gated,-0.01,"[-0.050, +0.030]",False
1,seen,steering_vs_lora,-0.33,"[-0.420, -0.250]",True
2,seen,steering_vs_linmap_lora,-0.10,"[-0.210, +0.010]",False
3,seen,steering_vs_linmap,-0.22,"[-0.310, -0.130]",True
4,seen,steering_vs_reft,0.08,"[-0.030, +0.190]",False
5,seen,steering_vs_cast,0.03,"[-0.040, +0.110]",False
6,unseen,steering_vs_steering_gated,0.06,"[+0.010, +0.110]",True
7,unseen,steering_vs_lora,-0.31,"[-0.420, -0.200]",True
8,unseen,steering_vs_linmap_lora,-0.24,"[-0.370, -0.120]",True
9,unseen,steering_vs_linmap,-0.07,"[-0.130, -0.020]",True


### Цена

In [7]:
f = exp.state["fitted"]
d = exp.backend.hidden_size
cost = [{"method": "steering (CAA)", "train": "среднее по 110 промптам", "seconds": None, "params": d}]
lr = f["lora"]["row"]
cost.append({"method": "lora", "train": "градиент по весам (копия fp32)", "seconds": lr["train_seconds"],
             "params": lr["n_trainable"]})
for m in ("linmap_lora", "linmap"):
    r, c = f[m]["row"], f[m].get("cost", {})
    if r:
        cost.append({"method": m, "train": "закрытая форма" + (" + учитель LoRA" if m == "linmap_lora" else ""),
                     "seconds": c.get("acts_seconds", 0) + c.get("fit_seconds", 0), "params": 2 * d * r["rank"] + d})
r = f["reft"]["row"]
if r:
    cost.append({"method": "reft", "train": "градиент только по вмешательству", "seconds": r["train_seconds"],
                 "params": r["n_trainable"]})
if f["cast"]["row"]:
    cost.append({"method": "cast", "train": "PC1 + порог по F1", "seconds": None, "params": 2 * d + 1})
pd.DataFrame(cost)

,method,train,seconds,params
0,steering (CAA),среднее по 110 промптам,NaN,2048
1,lora,градиент по весам (копия fp32),671.9,3211264
2,linmap_lora,закрытая форма + учитель LoRA,266.0,67584
3,linmap,закрытая форма,1.2,34816
4,reft,градиент только по вмешательству,590.4,16388
5,cast,PC1 + порог по F1,NaN,4097


## Вывод → журнал

In [8]:
display(Markdown(exp.report(
    conclusion="""РАЗВЕДКА (hypothesis=explore, test не открыт). Вмешательства в точке вызова против LoRA (учитель-маркер), Qwen3-1.7B, v3, одинаковые 110 промптов train (меняющие пары) для всех обучаемых методов.

1) Форма на слоях 22–26 (156 промптов train, 63 расхождения LoRA/база): линейное отображение состояния воспроизводит 68–76% исправлений LoRA (лучший L23: 0.762), сохраняя 97% совпавших решений; постоянный вектор — 5–25%, вектор по состоянию probe — 16–27%. На L18–20 (07) линейная форма давала 44–48%. Значимая для решения часть Δh на поздних слоях — линейная функция состояния, не постоянный сдвиг: это объясняет слабость CAA.

2) Перенос, новый набор seed 202 (успех / чувств. / специф.; эпизоды):
- известные типы: база 0.60/0.27/0.62 (эп. 0.56); LoRA 0.96/0.89/1.00 (0.84); linmap_lora 0.73/0.46/0.69 (0.68); linmap 0.85/0.59/1.00 (0.68); reft 0.55/0.11/0.46 (0.53); steering 0.63/0.22/0.69; cast 0.60/0.30/0.46.
- новые типы: база 0.45/0.12/0.25 (0.38); LoRA 0.85/0.62/0.88 (0.79); linmap_lora 0.78/0.44/0.81 (0.74); linmap 0.61/0.29/0.44 (0.39); reft 0.53/0.38/0.31 (0.52); steering 0.54/0.24/0.44; cast 0.49/0.18/0.38.
- linmap без учителя (закрытая форма, L18, ранг 8, α=1.5): на известных типах ~70% прироста LoRA без потери специфичности, в эпизодах +0.12 против +0.28; на новых типах прирост одношаговый +0.16, в эпизодах ноль. Цель «центроид верных состояний с тем же действием» запоминает типы требований train и не переносится.
- linmap_lora (учитель LoRA, L26, ранг 16, α=1.5): на новых типах 0.78 против 0.85 у LoRA, эпизоды 0.74 против 0.79. Аффинной поправки в одной позиции одного слоя достаточно, чтобы нести большую часть переносимого эффекта LoRA; узкое место варианта без учителя — цель, а не форма.
- ReFT (LoReFT ранг 4, градиент только по вмешательству, ~10 мин на слой): при наших настройках не лучше базы; гиперпараметры почти не подбирались.
- CAST: условие срабатывает на 92% промптов, разделения нет, эффект нулевой.

3) Цена (обучение без отбора на val): LoRA 672 с, 3.2M параметров; linmap — проход базы по 110 промптам (~2 мин, общий с linmap_lora) + 1.2 с подгонки, 35k параметров; linmap_lora — 266 с активаций + цена самой LoRA; ReFT 590 с, 16k параметров. Отбор на val у linmap — 54 конфигурации (~27 мин) против 2 у LoRA. У обоих linmap лучший α=1.5 на краю сетки.

Ограничения: val 31 задача; на переносе 34–37 меняющих и 13–16 инвариантных пар на часть; интервалы широкие. Всё — разведка, подтверждать на отложенной выборке.""",
    next_step="""1) Цель без учителя, которая переносится: (а) учитель-промпт — Δh = h(explicit_state) − h(база) в точке вызова, та же подгонка в закрытой форме (explicit_state на новых типах в 06 давал 0.85; градиента нет); (б) цель по считыванию решения — направление роста log p(верного действия) в точке вызова (один обратный проход на пример, без обучения весов). 2) Идея SimplexLoRA: несколько слоёв 22–26 с бюджетом ранга/α, распределённым по важности слоя на симплексе; важность без обучения — из проверки формы (доля исправлений по слою) или качества подгонки. 3) Расширить сетку α (1.5 на краю). 4) CAST в текущем виде снять; ReFT — только если нужен градиентный ориентир, с подбором lr/эпох.""",
)))

## 2026-10-31 00:43 · прогон `20261030-204700_act-methods-qwen3-1.7b_74c4d6`

**Гипотеза:** explore · **Ноутбук:** notebooks/08_activation_methods.ipynb · **Модель:** Qwen/Qwen3-1.7B (mps, float16) · **Режим решения:** scored
**Стадии:** data, baseline, probe, vectors, sweep, fit, diagnose, transfer · **git:** `bf93807` (есть незакоммиченные изменения)

**Зачем:** Вмешательства в точке вызова: linmap (с LoRA-учителем и без), LoReFT, CAST против LoRA.

**Отличия от конфига по умолчанию:** `data.n_tasks` = 160; `episode.on_val` = False; `eval.methods` = ['baseline', 'steering', 'steering_gated', 'lora', 'linmap_lora', 'linmap', 'reft', 'cast']; `model.name` = 'Qwen/Qwen3-1.7B'; `transfer.episode_methods` = ['baseline', 'lora', 'linmap_lora', 'linmap', 'reft']; `transfer.seed` = 202

**⚠ Флаги протокола:**
- незакоммиченные изменения в src/docs/notebooks/configs — прогон не воспроизводится по коммиту

**Результаты (val):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.621 | 0.526 | 0.400 | — |  | — | — | 831 | 10.000 | 400.600 |
| steering | 0.655 | 0.632 | 0.400 | 0.034 | [+0.000, +0.086] | 2.000 | 0.000 | 831 | 10.000 | 410.200 |
| steering_gated | 0.638 | 0.579 | 0.400 | 0.017 | [+0.000, +0.052] | 1.000 | 0.000 | 831 | 10.000 | 408.600 |
| lora | 0.948 | 1.000 | 0.800 | 0.328 | [+0.190, +0.466] | 19.000 | 0.000 | 831 | 10.000 | 299.000 |
| linmap_lora | 0.845 | 0.789 | 0.700 | 0.224 | [+0.086, +0.379] | 18.000 | 5.000 | 831 | 10.000 | 425.800 |
| linmap | 0.862 | 0.842 | 0.600 | 0.241 | [+0.121, +0.379] | 16.000 | 2.000 | 831 | 10.000 | 410.600 |
| reft | 0.741 | 0.684 | 0.500 | 0.121 | [-0.017, +0.259] | 13.000 | 6.000 | 831 | 10.000 | 442.900 |
| cast | 0.621 | 0.526 | 0.400 | 0.000 | [+0.000, +0.000] | 0.000 | 0.000 | 831 | 10.000 | 440.600 |

**Перенос, seen (разведка):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | episode | ep_diff | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.600 | 0.270 | 0.615 | — |  | — | — | 0.560 | — | 834 | 10.000 | 420.900 |
| steering | 0.630 | 0.216 | 0.692 | 0.030 | [-0.040, +0.110] | 8.000 | 5.000 | — | — | 834 | 10.000 | 414.700 |
| steering_gated | 0.640 | 0.270 | 0.692 | 0.040 | [-0.010, +0.110] | 5.000 | 1.000 | — | — | 834 | 10.000 | 416.100 |
| lora | 0.960 | 0.892 | 1.000 | 0.360 | [+0.260, +0.460] | 38.000 | 2.000 | 0.840 | 0.280 | 834 | 10.000 | 428.200 |
| linmap_lora | 0.730 | 0.459 | 0.692 | 0.130 | [+0.020, +0.240] | 26.000 | 13.000 | 0.680 | 0.120 | 834 | 10.000 | 424.000 |
| linmap | 0.850 | 0.595 | 1.000 | 0.250 | [+0.150, +0.360] | 32.000 | 7.000 | 0.680 | 0.120 | 834 | 10.000 | 426.900 |
| reft | 0.550 | 0.108 | 0.462 | -0.050 | [-0.160, +0.060] | 18.000 | 23.000 | 0.530 | -0.030 | 834 | 10.000 | 426.100 |
| cast | 0.600 | 0.297 | 0.462 | 0.000 | [-0.060, +0.050] | 3.000 | 3.000 | — | — | 834 | 10.000 | 850.700 |

**Перенос, unseen (разведка):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | episode | ep_diff | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.450 | 0.118 | 0.250 | — |  | — | — | 0.380 | — | 869 | 10.000 | 424.100 |
| steering | 0.540 | 0.235 | 0.438 | 0.090 | [+0.020, +0.170] | 12.000 | 3.000 | — | — | 869 | 10.000 | 409.400 |
| steering_gated | 0.480 | 0.118 | 0.312 | 0.030 | [-0.020, +0.080] | 5.000 | 2.000 | — | — | 869 | 10.000 | 415.500 |
| lora | 0.850 | 0.618 | 0.875 | 0.400 | [+0.290, +0.510] | 42.000 | 2.000 | 0.790 | 0.410 | 869 | 10.000 | 366.900 |
| linmap_lora | 0.780 | 0.441 | 0.812 | 0.330 | [+0.210, +0.450] | 41.000 | 8.000 | 0.740 | 0.360 | 869 | 10.000 | 355.900 |
| linmap | 0.610 | 0.294 | 0.438 | 0.160 | [+0.090, +0.240] | 17.000 | 1.000 | 0.390 | 0.010 | 869 | 10.000 | 357.400 |
| reft | 0.530 | 0.382 | 0.312 | 0.080 | [-0.010, +0.170] | 14.000 | 6.000 | 0.520 | 0.140 | 869 | 10.000 | 358.300 |
| cast | 0.490 | 0.176 | 0.375 | 0.040 | [-0.020, +0.100] | 6.000 | 2.000 | — | — | 869 | 10.000 | 764.900 |

**Вердикты (автоматически, по правилам HYPOTHESES.md):**
- H1: ✔ support — лучший probe L16/post_message: AUROC 0.980, acc 0.882 при перестановке 0.573
- H2: ✔ support — различает (H1), но ошибается в 47% меняющих пар: чувствительность 0.526, специфичность 0.400
- H3: ? inconclusive (val) — на val: успех 0.621 → 0.655 при L16/pre_action/α=1.0; решает только test
- H4: ✘ against (val) — LoRA не хуже steering на условном срезе: steering − LoRA -0.293 [-0.431, -0.155]
- ALT: · not_run — steering не сравнивался с промптом / пересказом / явным состоянием

**Вывод:** РАЗВЕДКА (hypothesis=explore, test не открыт). Вмешательства в точке вызова против LoRA (учитель-маркер), Qwen3-1.7B, v3, одинаковые 110 промптов train (меняющие пары) для всех обучаемых методов.

1) Форма на слоях 22–26 (156 промптов train, 63 расхождения LoRA/база): линейное отображение состояния воспроизводит 68–76% исправлений LoRA (лучший L23: 0.762), сохраняя 97% совпавших решений; постоянный вектор — 5–25%, вектор по состоянию probe — 16–27%. На L18–20 (07) линейная форма давала 44–48%. Значимая для решения часть Δh на поздних слоях — линейная функция состояния, не постоянный сдвиг: это объясняет слабость CAA.

2) Перенос, новый набор seed 202 (успех / чувств. / специф.; эпизоды):
- известные типы: база 0.60/0.27/0.62 (эп. 0.56); LoRA 0.96/0.89/1.00 (0.84); linmap_lora 0.73/0.46/0.69 (0.68); linmap 0.85/0.59/1.00 (0.68); reft 0.55/0.11/0.46 (0.53); steering 0.63/0.22/0.69; cast 0.60/0.30/0.46.
- новые типы: база 0.45/0.12/0.25 (0.38); LoRA 0.85/0.62/0.88 (0.79); linmap_lora 0.78/0.44/0.81 (0.74); linmap 0.61/0.29/0.44 (0.39); reft 0.53/0.38/0.31 (0.52); steering 0.54/0.24/0.44; cast 0.49/0.18/0.38.
- linmap без учителя (закрытая форма, L18, ранг 8, α=1.5): на известных типах ~70% прироста LoRA без потери специфичности, в эпизодах +0.12 против +0.28; на новых типах прирост одношаговый +0.16, в эпизодах ноль. Цель «центроид верных состояний с тем же действием» запоминает типы требований train и не переносится.
- linmap_lora (учитель LoRA, L26, ранг 16, α=1.5): на новых типах 0.78 против 0.85 у LoRA, эпизоды 0.74 против 0.79. Аффинной поправки в одной позиции одного слоя достаточно, чтобы нести большую часть переносимого эффекта LoRA; узкое место варианта без учителя — цель, а не форма.
- ReFT (LoReFT ранг 4, градиент только по вмешательству, ~10 мин на слой): при наших настройках не лучше базы; гиперпараметры почти не подбирались.
- CAST: условие срабатывает на 92% промптов, разделения нет, эффект нулевой.

3) Цена (обучение без отбора на val): LoRA 672 с, 3.2M параметров; linmap — проход базы по 110 промптам (~2 мин, общий с linmap_lora) + 1.2 с подгонки, 35k параметров; linmap_lora — 266 с активаций + цена самой LoRA; ReFT 590 с, 16k параметров. Отбор на val у linmap — 54 конфигурации (~27 мин) против 2 у LoRA. У обоих linmap лучший α=1.5 на краю сетки.

Ограничения: val 31 задача; на переносе 34–37 меняющих и 13–16 инвариантных пар на часть; интервалы широкие. Всё — разведка, подтверждать на отложенной выборке.

**Следующий шаг:** 1) Цель без учителя, которая переносится: (а) учитель-промпт — Δh = h(explicit_state) − h(база) в точке вызова, та же подгонка в закрытой форме (explicit_state на новых типах в 06 давал 0.85; градиента нет); (б) цель по считыванию решения — направление роста log p(верного действия) в точке вызова (один обратный проход на пример, без обучения весов). 2) Идея SimplexLoRA: несколько слоёв 22–26 с бюджетом ранга/α, распределённым по важности слоя на симплексе; важность без обучения — из проверки формы (доля исправлений по слою) или качества подгонки. 3) Расширить сетку α (1.5 на краю). 4) CAST в текущем виде снять; ReFT — только если нужен градиентный ориентир, с подбором lr/эпох.
